## 1. Configuration

In [ ]:
# =============================================================================
# NOTEBOOK CONFIGURATION
# =============================================================================
# This cell contains all configurable parameters for the notebook.
# Modify these settings to customize the graph creation workflow.

# --- General Workflow Flags ---
# Toggle major workflow steps on/off for testing or partial runs
backend = "geopackage" # Postgis, All
graph_name = "fine_graph_test_graph" # Source (undirected) graph
grid_name = f"{graph_name}_grids"
save_output = True              # Save graph to GeoPackage file

# --- Data Sources & Paths ---
# PostGIS schema and table names for input data
db_schema_enc = "enc_west"        # Schema containing S-57 ENC layers
db_schema_graph = "graph"
db_schema_grid = "grid"
data_file_name = "enc_west.gpkg"  # GeoPackage filename in output/ directory
config_file_path = "src/nautical_graph_toolkit/data/graph_config.yml"  # YAML config for layers/H3 settings


# --- Geographic & Route Parameters ---
buffer_size_nm = 24.0           # Buffer distance around route in nautical miles
slice_south_degree = 37.0             # Southern latitude limit for buffer slicing (if buffer_sliced=True)


# --- Configuration Summary ---
print("=" * 70)
print("✓ Configuration loaded successfully!")
print("=" * 70)
print(f"📍 Graph name: {graph_name}")
print(f"📐 Buffer expansion: {buffer_size_nm} NM")
print("=" * 70)

### 1.2 Imports

In [ ]:
# =============================================================================
# SECTION 1: IMPORTS AND CONFIGURATION
# =============================================================================

import os
import sys
from pathlib import Path

import geopandas as gpd
import networkx as nx
import pandas as pd
from shapely.geometry import Point, LineString, Polygon, box
import plotly.graph_objects as go
from dotenv import load_dotenv

# --- Fix PROJ_LIB Path (Common Conda/Jupyter Issue) ---
conda_prefix = sys.prefix
possible_proj_lib = os.path.join(conda_prefix, 'share', 'proj')
if os.path.exists(possible_proj_lib):
      os.environ['PROJ_LIB'] = possible_proj_lib

# --- Setup Python Environment ---
# Get project root for .env file loading
project_root = Path.cwd().parent.parent
assert (project_root / "src" / "nautical_graph_toolkit").exists()

from nautical_graph_toolkit.core.s57_data import ENCDataFactory
from nautical_graph_toolkit.utils.geometry_utils import Grid, Buffer
from nautical_graph_toolkit.utils.plot_utils import PlotlyChart
from nautical_graph_toolkit.core.weights import Weights
from nautical_graph_toolkit.core.weight_calculator import WeightCalculator
from nautical_graph_toolkit.utils.notebook_utils import BenchmarkLogger

# --- Setup Python Environment ---
project_root = Path.cwd().parent.parent
load_dotenv(project_root / ".env")

# --- Output Directory ---
output_dir = Path.cwd() / 'output'
data_dir = project_root / 'data'
output_dir.mkdir(exist_ok=True)

# --- Define Data Source (PostGIS Database Connection) ---
# PostGIS uses dict of connection params loaded from environment
# These credentials are loaded from the .env file
db_params = {
    'dbname': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT')
}

# Define database file
data_file = project_root / "data" / data_file_name

GPKG_GRAPH_PATH = f"{data_dir}/{graph_name}.gpkg"

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")
print(f"GeoPackage Graph path: {GPKG_GRAPH_PATH}")
print(f"PostGIS DB_name: {db_params['dbname']}")

logger = BenchmarkLogger()




### 1.3 Workflow Context

**Purpose**: Demonstrates and validates the `Buffer` and `Grid` utility classes across GeoPackage and PostGIS backends.

**Pipeline Position**: Supporting utility — use alongside any graph workflow notebook.

**Prerequisites**:
- A pre-built graph saved to GeoPackage (`GPKG_GRAPH_PATH`)
- ENC data available as GeoPackage (`ENC_GPKG_PATH`) or PostGIS (`POSTGIS_DB_PARAMS`)

**Outputs**:
- Buffer polygon around graph edges
- Progressive grid geometries (combined, main, land)
- Comparison metrics between GeoPackage and PostGIS backends
- Exported GeoPackage: `output/grid_test_geopkg.gpkg`
- Comparison plot: `output/grid_comparison.png`

**Related Notebooks**:
- `graph_fine_GeoPackage_v2.ipynb` — full fine graph workflow (GeoPackage)
- `graph_fine_PostGIS_v2.ipynb` — full fine graph workflow (PostGIS)

### 1.4. Initialize PostGIS Data Factory

In [ ]:
# --- Initialize ENC Data Factory for PostGIS Backend ---
# The factory provides a unified interface for accessing ENC data
# from PostGIS database. It will be used throughout the notebook
# for querying navigational layers and saving results.
pg_factory = ENCDataFactory(source=db_params, schema=db_schema_enc)

# --- Initialize ENC Data Factory for GeoPackage Backend ---
# The factory provides a unified interface for accessing ENC data
# from GeoPackage. It will be used throughout the notebook
# for querying navigational layers and saving results.
gpkg_factory = ENCDataFactory(source=data_file)

### 1.6 Initialize Plotly Visualization

In [ ]:
# --- Create Base Plotly Map ---
# Initialize the plotting utility and create an interactive base map
# The Mapbox token is loaded from .env for security (not hardcoded)
# The base map will be copied for different visualizations throughout the notebook
ply = PlotlyChart()
ply_fig = ply.create_base_map(mapbox_token=os.getenv('MAPBOX_TOKEN'))
ply.plotly_base_config(ply_fig)

In [ ]:
# =============================================================================
# SECTION 2: LOAD GRAPH FROM BACKEND
# =============================================================================

edges_gdf = gpd.read_file(GPKG_GRAPH_PATH, layer='edges')
# Simplest bounding box outline
gpkg_bounds = box(*edges_gdf.total_bounds)
gpkg_buffer = Buffer.create_buffer(gpkg_bounds, buffer_size_nm)


In [ ]:
# --- Visualize Buffer on Map ---
# Add the buffer polygon to the map to verify it covers the desired area
ply_gpkg = go.Figure(ply_fig)
ply.add_polygon_trace(fig=ply_gpkg,
                      polygon=gpkg_bounds,
                      name="Graph Bounds",
                      color='blue')
ply.add_polygon_trace(fig=ply_gpkg,
                      polygon=gpkg_buffer,
                      name="Graph Buffer",
                      color='red')
ply_gpkg.show()

In [ ]:
enc_list_gpkg = gpkg_factory.get_encs_by_boundary(gpkg_buffer)
print(f'Found {len(enc_list_gpkg)} ENCs')

In [ ]:
from shapely import wkt as shapely_wkt
from sqlalchemy import text
# PostGIS native — query bounding box without loading all edges
with pg_factory.manager.engine.connect() as conn:
    result = conn.execute(text(
      f"SELECT ST_AsText(ST_Envelope(ST_Extent(geometry))) "
      f"FROM {db_schema_graph}.{graph_name}_edges"
    ))
    bbox_wkt = result.scalar()

pg_bounds = shapely_wkt.loads(bbox_wkt)
pg_buffer = Buffer.create_buffer(pg_bounds, buffer_size_nm)

In [ ]:
# --- Visualize Buffer on Map ---
# Add the buffer polygon to the map to verify it covers the desired area
ply_pg = go.Figure(ply_fig)
ply.add_polygon_trace(fig=ply_pg,
                      polygon=pg_bounds,
                      name="Graph Bounds",
                      color='blue')
ply.add_polygon_trace(fig=ply_pg,
                      polygon=pg_buffer,
                      name="Graph Buffer",
                      color='red')
ply_pg.show()

In [ ]:
enc_list_pg = pg_factory.get_encs_by_boundary(pg_buffer)
print(f'Found {len(enc_list_pg)} ENCs')
enc_list_pg

In [ ]:
# Same elements, any order
set(enc_list_gpkg) == set(enc_list_pg)

In [ ]:
type(enc_list_gpkg)

In [ ]:
# =============================================================================
# SECTION 5: CREATE GRIDS USING GRID CLASS (GEOPACKAGE BACKEND)
# =============================================================================
logger.start_timer('grid_creation_gpkg')
nav_layers = [{"layer": "seaare", "bands": "all"}]
obstn_layers = [{"layer": "lndare", "bands": "all"}]
gpkg_grid_name = f"{grid_name}.gpkg"

grid_gpkg_result = Grid.progressive_grid(
      buffer=gpkg_buffer,
      factory=gpkg_factory,
      enc_names=enc_list_gpkg,
      navigable_layers=nav_layers,
      obstacle_layers=obstn_layers,
  )
elapsed = logger.end_step('grid_creation_gpkg')

print("GeoPackage Grid Results:")
print(f"  Combined grid: {grid_gpkg_result['combined_grid_geom'].geom_type}")
print(f"  Main grid:     {grid_gpkg_result['main_grid_geom'].geom_type}")
print(f"  Land grid:     {grid_gpkg_result['land_grid_geom'].geom_type}")
print(f"  Grid creation took: {elapsed:.2f}s")

Grid.save_to_gpkg(
      grid_gpkg_result,
      output_path=str(output_dir / gpkg_grid_name),
      layers=['combined_grid', 'main_grid', 'land_grid', 'extra_grid', 'subtract_grid']
  )
print(f"Exported GeoPackage grids to {output_dir / gpkg_grid_name}")

In [ ]:
ply_gpkg_grid = go.Figure(ply_fig)
ply.add_grid_trace(ply_gpkg_grid, name="Main Grid", grid_geojson=grid_gpkg_result["main_grid"], color="blue")
ply.add_grid_trace(ply_gpkg_grid, name="Combined Grid", grid_geojson=grid_gpkg_result["combined_grid"], color="red")
if grid_gpkg_result["extra_grid"] is not None:
  ply.add_grid_trace(ply_gpkg_grid, name="Extra Grid", grid_geojson=grid_gpkg_result["extra_grid"], color="yellow")
if grid_gpkg_result["land_grid"] is not None:
  ply.add_grid_trace(ply_gpkg_grid, name="Land Grid", grid_geojson=grid_gpkg_result["land_grid"], color="green")
ply_gpkg_grid.show()

In [ ]:
# =============================================================================
# SECTION 6: CREATE GRIDS USING GRID CLASS (POSTGIS BACKEND)
# =============================================================================

logger.start_timer('grid_creation_postgis')
grid_postgis_result = Grid.progressive_grid(
      buffer=pg_buffer,
      factory=pg_factory,
      enc_names=enc_list_pg,
      navigable_layers=nav_layers,
      obstacle_layers=obstn_layers,
  )
elapsed = logger.end_step('grid_creation_postgis')

print("PostGIS Grid Results:")
print(f"  Combined grid: {grid_postgis_result['combined_grid_geom'].geom_type}")
print(f"  Main grid:     {grid_postgis_result['main_grid_geom'].geom_type}")
print(f"  Land grid:     {grid_postgis_result['land_grid_geom'].geom_type}")
print(f"  Grid creation took: {elapsed:.2f}s")


In [ ]:
ply_pg_grid = go.Figure(ply_fig)
ply.add_grid_trace(ply_pg_grid, name="Main Grid", grid_geojson=grid_postgis_result["main_grid"], color="blue")
ply.add_grid_trace(ply_pg_grid, name="Combined Grid", grid_geojson=grid_postgis_result["combined_grid"], color="red")
if grid_gpkg_result["extra_grid"] is not None:
  ply.add_grid_trace(ply_pg_grid, name="Extra Grid", grid_geojson=grid_postgis_result["extra_grid"], color="yellow")
if grid_gpkg_result["land_grid"] is not None:
  ply.add_grid_trace(ply_pg_grid, name="Land Grid", grid_geojson=grid_postgis_result["land_grid"], color="green")
ply_pg_grid.show()

In [ ]:
# Visual comparison of Geopackage and Postgis Grids
ply_grid_compare = go.Figure(ply_fig)
ply.add_grid_trace(ply_grid_compare, name="GeoPackage Land Grid", grid_geojson=grid_gpkg_result["land_grid"], color="yellow")
ply.add_grid_trace(ply_grid_compare, name="PostGIS Land Grid", grid_geojson=grid_postgis_result["land_grid"], color="green")
ply_grid_compare.show()

In [ ]:
Grid.save_to_postgis(
      grid_postgis_result,
      table_name=grid_name,
      schema=db_schema_grid,
      connection_params=db_params,
      layers=['combined_grid', 'main_grid', 'land_grid']
  )
print(f"Exported PostGIS grids to {db_schema_grid}.{grid_name} tables")

In [ ]:
from tests.core__real_data.test_buffer_geometry_utils import compare_graph_to_geometry, assert_backends_agree

In [ ]:


grid_file_path = project_root / "docs" / "notebooks" / "output" / "fine_graph_test_graph_grids.gpkg"
graph_file_path = project_root / "data" / "fine_graph_test_graph.gpkg"

# Load your data
edges = gpd.read_file(graph_file_path, layer="edges")
feats = gpd.read_file(grid_file_path, layer="land_grid")

print(feats.geometry.apply(lambda g: len(g.exterior.coords) if hasattr(g, 'exterior') else sum(len(p.exterior.coords) for p in g.geoms)).describe())

if edges.empty:
    print("No edges found")
if feats.empty:
    print("No features provided")

try:
    edges = gpd.read_file(graph_file_path, layer="edges")
    feats = gpd.read_file(grid_file_path, layer="land_grid")
except Exception as e:
    print(f"Load failed: {e}")
else:
    print(f"edges: {len(edges)} rows, feats: {len(feats)} rows")

print(f"edges: {len(edges):,} rows, CRS: {edges.crs}")
print(f"feats: {len(feats):,} rows, CRS: {feats.crs}")
print(f"feats geometry types: {feats.geom_type.value_counts().to_dict()}")

feats_simple = feats.copy()
feats_simple.geometry = feats.geometry.simplify(tolerance=0.001, preserve_topology=True)
print(f"Before: {feats.geometry.apply(lambda g: g.length).sum():.0f}")
print(f"After:  {feats_simple.geometry.apply(lambda g: g.length).sum():.0f}")


# Run all backends
results = compare_graph_to_geometry(
      edges_gdf=edges,
      feats_gdf=feats_simple,
      layer_name="land_grid",
      strategy="fast",
      buffer_m=10,                              # override the 500 m default
      graph_gpkg=Path(graph_file_path),
      enc_gpkg=Path(grid_file_path),
      postgis_engine=None,                         # skip PostGIS
  )

# Inspect
for backend, (count, elapsed) in results.items():
      print(f"{backend:10s}  count={count}  time={elapsed:.2f}s")

# Assert consistency (optional)
assert_backends_agree(results, label="land/fast")

In [ ]:
# =============================================================================
# SECTION 7: FILTER LAND GRID INTERSECTION
# =============================================================================

def identify_land_intersecting_edges(
     edges_gdf: gpd.GeoDataFrame,
     land_grid_geom: Polygon,
     backend: str = 'geopandas'
 ) -> gpd.GeoDataFrame:
     """
     Identify edges that intersect with land grid.

     Mirrors the _identify_land_intersecting_edges_geopandas method
     from weights.py (lines 3967-4008).

     Args:
         edges_gdf: GeoDataFrame of graph edges
         land_grid_geom: Land grid geometry from Grid.progressive_grid()
         backend: 'geopandas' or 'postgis'

     Returns:
         GeoDataFrame: Edges that intersect land
     """
     if backend == 'geopandas':
         # Pure Shapely operation (in-memory)
         land_union = land_grid_geom
         intersecting_mask = edges_gdf.geometry.intersects(land_union)
         land_edges = edges_gdf[intersecting_mask].copy()
         land_edges['intersects_land'] = True
     else:
         # PostGIS backend would use ST_Intersects
         # For now, use geopandas approach
         land_union = land_grid_geom
         intersecting_mask = edges_gdf.geometry.intersects(land_union)
         land_edges = edges_gdf[intersecting_mask].copy()
         land_edges['intersects_land'] = True

     return land_edges

# Test with GeoPackage land grid
land_edges_gpkg = identify_land_intersecting_edges(
     edges_gdf,
     grid_gpkg_result['land_grid_geom'],
     backend='geopandas'
 )

print(f"GeoPackage: {len(land_edges_gpkg):,} edges intersect land ({len(land_edges_gpkg)/len(edges_gdf)*100:.1f}%)")

 # Test with PostGIS land grid
 land_edges_postgis = identify_land_intersecting_edges(
     edges_gdf,
     grid_postgis_result['land_grid_geom'],
     backend='geopandas'
 )

 print(f"PostGIS: {len(land_edges_postgis):,} edges intersect land ({len(land_edges_postgis)/len(edges_gdf)*100:.1f}%)")


In [ ]:
# =============================================================================
# SECTION 8: COMPARE BACKEND RESULTS
# =============================================================================

def compare_grids(grid_gpkg: dict, grid_postgis: dict, edges_gdf: gpd.GeoDataFrame) -> dict:
     """
     Compare grid results between GeoPackage and PostGIS backends.

     Args:
         grid_gpkg: Result dict from GeoPackage backend
         grid_postgis: Result dict from PostGIS backend
         edges_gdf: Original edges for intersection testing

     Returns:
         dict: Comparison metrics
     """
     import json

     results = {}

     # Compare geometry areas
     results['combined_area_diff_pct'] = (
         abs(grid_gpkg['combined_grid_geom'].area - grid_postgis['combined_grid_geom'].area) /
         grid_gpkg['combined_grid_geom'].area * 100
     )
     results['land_area_diff_pct'] = (
         abs(grid_gpkg['land_grid_geom'].area - grid_postgis['land_grid_geom'].area) /
         grid_gpkg['land_grid_geom'].area * 100
     )

     # Compare land intersection counts
     land_gpkg = identify_land_intersecting_edges(
         edges_gdf, grid_gpkg['land_grid_geom'], 'geopandas'
     )
     land_postgis = identify_land_intersecting_edges(
         edges_gdf, grid_postgis['land_grid_geom'], 'geopandas'
     )

     results['land_intersect_count_gpkg'] = len(land_gpkg)
     results['land_intersect_count_postgis'] = len(land_postgis)
     results['land_intersect_diff'] = abs(len(land_gpkg) - len(land_postgis))
     results['land_intersect_diff_pct'] = (
         results['land_intersect_diff'] / len(edges_gdf) * 100
     )

     # Check if geometries are "nearly equal" (accounting for floating point)
     results['combined_equal'] = grid_gpkg['combined_grid_geom'].equals_exact(
         grid_postgis['combined_grid_geom'], tolerance=1e-6
     )
     results['land_equal'] = grid_gpkg['land_grid_geom'].equals_exact(
         grid_postgis['land_grid_geom'], tolerance=1e-6
     )

     return results

# Run comparison
comparison = compare_grids(grid_gpkg_result, grid_postgis_result, edges_gdf)

print("=== Backend Comparison Results ===")
print(f"Combined grid area difference: {comparison['combined_area_diff_pct']:.4f}%")
print(f"Land grid area difference: {comparison['land_area_diff_pct']:.4f}%")
print(f"Land intersecting edges - GPKG: {comparison['land_intersect_count_gpkg']:,}")
print(f"Land intersecting edges - PostGIS: {comparison['land_intersect_count_postgis']:,}")
print(f"Land intersect difference: {comparison['land_intersect_diff']} edges ({comparison['land_intersect_diff_pct']:.2f}%)")
print(f"Geometries exactly equal (combined): {comparison['combined_equal']}")
print(f"Geometries exactly equal (land): {comparison['land_equal']}")

In [ ]:
# =============================================================================
# SECTION 9: VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt

def plot_grid_comparison(
     edges_gdf: gpd.GeoDataFrame,
     grid_gpkg_result: dict,
     grid_postgis: dict,
     land_edges_gpkg: gpd.GeoDataFrame = None,
     land_edges_postgis: gpd.GeoDataFrame = None
 ):
     """
     Create comparison plot of grids from both backends.
     """
     fig, axes = plt.subplots(1, 2, figsize=(16, 8))

     # GeoPackage plot
     ax = axes[0]
     edges_gdf.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.5, label='Edges')
     gpd.GeoSeries([grid_gpkg_result['combined_grid_geom']]).plot(
         ax=ax, facecolor='blue', alpha=0.3, edgecolor='none', label='Navigable'
     )
     gpd.GeoSeries([grid_gpkg_result['land_grid_geom']]).plot(
         ax=ax, facecolor='red', alpha=0.3, edgecolor='none', label='Land'
     )
     if land_edges_gpkg is not None and len(land_edges_gpkg) > 0:
         land_edges_gpkg.plot(ax=ax, color='orange', linewidth=1, label='Land intersects')
     ax.set_title('GeoPackage Backend')
     ax.legend()
     ax.set_aspect('equal')

     # PostGIS plot
     ax = axes[1]
     edges_gdf.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.5, label='Edges')
     gpd.GeoSeries([grid_postgis['combined_grid_geom']]).plot(
         ax=ax, facecolor='blue', alpha=0.3, edgecolor='none', label='Navigable'
     )
     gpd.GeoSeries([grid_postgis['land_grid_geom']]).plot(
         ax=ax, facecolor='red', alpha=0.3, edgecolor='none', label='Land'
     )
     if land_edges_postgis is not None and len(land_edges_postgis) > 0:
         land_edges_postgis.plot(ax=ax, color='orange', linewidth=1, label='Land intersects')
     ax.set_title('PostGIS Backend')
     ax.legend()
     ax.set_aspect('equal')

     plt.tight_layout()
     plt.savefig('../output/grid_comparison.png', dpi=150)
     plt.show()


In [ ]:
# =============================================================================
# SECTION 10: PERFORMANCE SUMMARY
# =============================================================================

csv_path = logger.export_benchmark()
print(f"\n💾 Benchmark saved to: {csv_path}")
print(logger.get_current_benchmark_summary())

In [ ]:
# Generate comparison plot
plot_grid_comparison(
     edges_gdf,
     grid_gpkg_result,
     grid_postgis_result,
     land_edges_gpkg,
     land_edges_postgis
 )